# Interactive frame–segment bipartite network

This notebook is separate from the main analysis notebook and is dedicated to building/exporting the interactive **document-segment ↔ frame type** network.

It preserves segment-level metadata and supports filters for model, document, year range, number of frames (`frame_label`), frame name (multi-select), polarity, relevance to original topic, topic type (multi-select), gender mentioned, and minimum frame degree. It also includes optional segment labels, optional extended metadata on hover, physics controls, performance safeguards, and HTML export for GitHub Pages.

The network uses two tables:

- `segment_df` from the usable (`ok`) rows, so `NOT_FRAMED` segments and metadata remain available;
- `edge_df` from exploded `frame_type`, one row per segment × frame type.

In [ ]:
import glob
import html as html_lib
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

FRAME_NAMES = [
    "modernization", "patriotism", "victimhood",
    "moral", "economic", "diplomacy",
]
POLARITIES = ["positive", "negative"]
FRAME_LABEL_ORDER = ["NOT_FRAMED", "SINGLE_FRAME", "MULTIPLE_FRAMES"]

FRAME_COLORS = {
    "modernization": "#4C78A8",
    "patriotism": "#F58518",
    "victimhood": "#E45756",
    "moral": "#72B7B2",
    "economic": "#54A24B",
    "diplomacy": "#B279A2",
}

DATA_DIR = Path("./data")
OUTPUT_FILE = Path("./frame_segment_network.html")

# Optional manual override, e.g. {"model name": "path/to/file.csv"}
MANUAL_FILES: dict[str, str] = {}

MAX_EDGES = 10_000
MAX_VISIBLE_SEGMENTS = 15_000

print("Data directory:", DATA_DIR.resolve())
print("HTML output:", OUTPUT_FILE.resolve())

In [ ]:
# ============================================================
# 1. Discover + load flattened frame-detection CSVs
# ============================================================

def discover_frame_csvs(data_dir: Path) -> list[Path]:
    patterns = [
        str(data_dir / "frames_*.csv"),
        str(data_dir / "frame_model_comparison" / "*" / "frames_*.csv"),
        str(data_dir / "frame_model_comparison" / "*" / "*.csv"),
    ]
    found = []
    for pat in patterns:
        found.extend(Path(p) for p in glob.glob(pat))
    seen, unique = set(), []
    for p in found:
        rp = p.resolve()
        if rp not in seen:
            seen.add(rp)
            unique.append(p)
    return sorted(unique)


def model_name_from_path(path: Path) -> str:
    if path.parent.name and path.parent.name != DATA_DIR.name:
        return path.parent.name
    m = re.search(r"frames_.+?_([A-Za-z0-9._-]+)\.csv$", path.name)
    return m.group(1) if m else path.stem


if MANUAL_FILES:
    discovered = [Path(p) for p in MANUAL_FILES.values()]
else:
    discovered = discover_frame_csvs(DATA_DIR)

assert discovered, "No frame-detection CSVs found. Change DATA_DIR or fill MANUAL_FILES."

print(f"Discovered {len(discovered)} CSV file(s):")
for p in discovered:
    print(" ", p)

frames = []
for p in discovered:
    d = pd.read_csv(p, keep_default_na=False, dtype=str)
    d["_source_file"] = str(p)
    frames.append(d)

raw = pd.concat(frames, ignore_index=True, sort=False)

if "framing_model" in raw.columns and raw["framing_model"].astype(str).str.strip().ne("").any():
    raw["model"] = raw["framing_model"].where(
        raw["framing_model"].astype(str).str.strip().ne(""),
        raw["_source_file"].apply(lambda s: model_name_from_path(Path(s))),
    )
else:
    raw["model"] = raw["_source_file"].apply(lambda s: model_name_from_path(Path(s)))

print(f"\nLoaded {len(raw):,} rows")
print(raw["model"].value_counts())

In [ ]:
# ============================================================
# 2. Build segment_df + edge_df
# ============================================================

if "frame_processing_error" in raw.columns:
    is_error = raw["frame_processing_error"].astype(str).str.upper().eq("TRUE")
    ok = raw.loc[~is_error].copy()
    print(f"Dropped {int(is_error.sum()):,} error row(s)")
else:
    ok = raw.copy()

required = ["doc_id", "segment_index", "frame_label", "frame_type", "model"]
missing = [c for c in required if c not in ok.columns]
assert not missing, f"Missing required columns: {missing}"

ok["segment_id"] = ok["doc_id"].astype(str) + "__" + ok["segment_index"].astype(str)

DATE_COLUMN_CANDIDATES = [
    "date", "pub_date", "publish_date", "publication_date",
    "article_date", "issue_date", "issue", "year",
]
date_col = next((c for c in DATE_COLUMN_CANDIDATES if c in ok.columns), None)
if date_col == "year":
    ok["_date"] = pd.to_datetime(ok["year"], format="%Y", errors="coerce")
elif date_col is not None:
    ok["_date"] = pd.to_datetime(ok[date_col], errors="coerce")
else:
    ok["_date"] = pd.NaT
ok["_year"] = ok["_date"].dt.year

metadata_cols = [
    "title", "frame_explanation", "frame_evidence",
    "relevance_to_original_topic", "topic_type", "gender_mentioned",
]
for col in metadata_cols:
    if col not in ok.columns:
        ok[col] = ""
    ok[col] = ok[col].fillna("").astype(str).str.strip()

ok["frame_label"] = ok["frame_label"].fillna("").astype(str).str.strip()

def split_semicolon_values(value):
    if pd.isna(value) or str(value).strip() == "":
        return []
    return [x.strip() for x in str(value).split(";") if x.strip()]

ok["topic_type_list"] = ok["topic_type"].apply(split_semicolon_values)

segment_df = (
    ok[[
        "segment_id", "doc_id", "segment_index", "model", "_year",
        "frame_label", "title", "frame_explanation", "frame_evidence",
        "relevance_to_original_topic", "topic_type", "topic_type_list",
        "gender_mentioned",
    ]]
    .drop_duplicates(subset=["segment_id", "model"])
    .rename(columns={"_year": "year"})
    .copy()
)

ok["frame_type"] = ok["frame_type"].fillna("").astype(str)
exploded = ok.assign(frame_type_single=ok["frame_type"].str.split("|")).explode("frame_type_single")
exploded["frame_type_single"] = exploded["frame_type_single"].fillna("").astype(str).str.strip()
exploded = exploded[exploded["frame_type_single"] != ""].copy()

def split_frame_polarity(value: str):
    m = re.match(r"^(.*)\s\((positive|negative)\)$", value)
    return (m.group(1), m.group(2)) if m else (value, None)

_split = exploded["frame_type_single"].apply(split_frame_polarity)
exploded["frame_name"] = _split.apply(lambda t: t[0])
exploded["polarity"] = _split.apply(lambda t: t[1])
exploded["frame_type"] = exploded["frame_type_single"]

edge_df = (
    exploded[["segment_id", "model", "frame_name", "polarity", "frame_type"]]
    .dropna(subset=["segment_id", "model", "frame_name", "polarity"])
    .drop_duplicates(subset=["segment_id", "model", "frame_type"])
    .copy()
)

MODEL_ORDER = sorted(segment_df["model"].astype(str).unique().tolist())

print(f"Segments/model rows: {len(segment_df):,}")
print(f"Frame links: {len(edge_df):,}")
print("Models:", MODEL_ORDER)
print(segment_df["frame_label"].value_counts(dropna=False))

In [ ]:
# ============================================================
# 3. Inspect retained metadata
# ============================================================

for col in [
    "title", "frame_explanation", "frame_evidence",
    "relevance_to_original_topic", "topic_type", "gender_mentioned",
]:
    nonempty = segment_df[col].astype(str).str.strip().ne("").sum()
    print(f"{col:30s} {nonempty:,}/{len(segment_df):,} non-empty")

print("\nTopic types:", sorted({t for values in segment_df["topic_type_list"] for t in values}))

In [ ]:
# ============================================================
# 4. Export interactive network HTML
# ============================================================

def _js_json(obj):
    return json.dumps(obj, ensure_ascii=False).replace("</", "<\\/")


def _esc(value):
    return html_lib.escape(str(value), quote=True)


def build_network_html(segment_df, edge_df, output_file=OUTPUT_FILE,
                       max_edges=MAX_EDGES,
                       max_visible_segments=MAX_VISIBLE_SEGMENTS):
    segment_records = []
    for _, row in segment_df.iterrows():
        year = None if pd.isna(row["year"]) else int(row["year"])
        segment_records.append({
            "segment_id": str(row["segment_id"]),
            "doc_id": str(row["doc_id"]),
            "segment_index": str(row["segment_index"]),
            "model": str(row["model"]),
            "year": year,
            "frame_label": str(row.get("frame_label", "")),
            "title": str(row.get("title", "")),
            "frame_explanation": str(row.get("frame_explanation", "")),
            "frame_evidence": str(row.get("frame_evidence", "")),
            "relevance_to_original_topic": str(row.get("relevance_to_original_topic", "")),
            "topic_types": list(row.get("topic_type_list", []) or []),
            "gender_mentioned": str(row.get("gender_mentioned", "")),
        })

    edge_records = [
        {
            "segment_id": str(row["segment_id"]),
            "model": str(row["model"]),
            "frame_name": str(row["frame_name"]),
            "polarity": str(row["polarity"]),
            "frame_type": str(row["frame_type"]),
        }
        for _, row in edge_df.iterrows()
    ]

    documents = sorted(segment_df["doc_id"].astype(str).unique().tolist())
    models = [m for m in MODEL_ORDER if m in segment_df["model"].astype(str).unique()]
    frame_names = [f for f in FRAME_NAMES if f in edge_df["frame_name"].astype(str).unique()]
    polarities = [p for p in POLARITIES if p in edge_df["polarity"].astype(str).unique()]

    valid_years = pd.to_numeric(segment_df["year"], errors="coerce").dropna().astype(int)
    year_min = int(valid_years.min()) if len(valid_years) else 0
    year_max = int(valid_years.max()) if len(valid_years) else 0

    frame_labels_present = set(segment_df["frame_label"].astype(str).unique())
    frame_labels = [x for x in FRAME_LABEL_ORDER if x in frame_labels_present]
    topic_values = sorted({t for vals in segment_df["topic_type_list"] for t in vals})
    has_missing_topic = segment_df["topic_type_list"].apply(len).eq(0).any()
    relevance_values = sorted(x for x in segment_df["relevance_to_original_topic"].astype(str).unique() if x.strip())
    gender_values = sorted(x for x in segment_df["gender_mentioned"].astype(str).unique() if x.strip())

    model_options = "".join(
        f'<option value="{_esc(m)}"' + (' selected' if i == 0 else '') + f'>{_esc(m)}</option>'
        for i, m in enumerate(models)
    )
    doc_options = "".join(f'<option value="{_esc(x)}">{_esc(x)}</option>' for x in documents)
    polarity_options = "".join(f'<option value="{_esc(x)}">{_esc(x)}</option>' for x in polarities)
    relevance_options = "".join(f'<option value="{_esc(x)}">{_esc(x)}</option>' for x in relevance_values)
    gender_options = "".join(f'<option value="{_esc(x)}">{_esc(x)}</option>' for x in gender_values)
    frame_checks = "".join(f'<label class="check-item"><input type="checkbox" value="{_esc(x)}" checked><span>{_esc(x)}</span></label>' for x in frame_names)
    label_checks = "".join(f'<label class="check-item"><input type="checkbox" value="{_esc(x)}" checked><span>{_esc(x)}</span></label>' for x in frame_labels)
    topic_checks = "".join(f'<label class="check-item"><input type="checkbox" value="{_esc(x)}" checked><span>{_esc(x)}</span></label>' for x in topic_values)
    if has_missing_topic:
        topic_checks += '<label class="check-item"><input type="checkbox" value="__MISSING__" checked><span>(missing)</span></label>'

    template = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Frame-segment bipartite network</title>
<link rel="stylesheet" href="https://unpkg.com/vis-network@9.1.9/styles/vis-network.min.css">
<script src="https://unpkg.com/vis-network@9.1.9/standalone/umd/vis-network.min.js"></script>
<style>
*{box-sizing:border-box}body{margin:0;font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Arial,sans-serif;color:#222}.page{display:grid;grid-template-columns:330px 1fr;height:100vh}.sidebar{padding:16px;overflow-y:auto;border-right:1px solid #ddd;background:#fafafa}.main{min-width:0;display:flex;flex-direction:column;height:100vh}#network{width:100%;height:calc(100vh - 42px);min-height:650px}h1{font-size:18px;margin:0 0 14px}h2{font-size:12px;margin:18px 0 7px;text-transform:uppercase;letter-spacing:.05em;color:#555}label{font-size:12px}select,input,button{font:inherit}select,input[type=number]{width:100%;min-height:36px;border:1px solid #ccc;border-radius:4px;padding:6px;background:#fff;margin-bottom:7px}button{border:1px solid #aaa;border-radius:4px;background:#fff;padding:6px 8px;cursor:pointer}.stats{padding:8px 12px;min-height:42px;border-bottom:1px solid #ddd;background:#fafafa;font-size:13px}.physics-grid{display:grid;grid-template-columns:1fr 1fr;gap:6px}.check-group{max-height:150px;overflow-y:auto;border:1px solid #ddd;border-radius:4px;padding:6px 8px;background:#fff}.check-item,.inline-check{display:flex;align-items:center;gap:7px;margin:5px 0}.check-item input,.inline-check input{width:auto;margin:0}.small-actions{display:flex;gap:5px;margin-bottom:5px}.small-actions button{flex:1;font-size:11px;padding:4px 6px}.details{margin-top:8px;padding:10px;border:1px solid #ddd;background:#fff;font-size:12px;line-height:1.45;min-height:45px;word-break:break-word}.legend-item{display:flex;align-items:center;margin:5px 0;font-size:12px}.swatch{width:13px;height:13px;border-radius:50%;margin-right:7px;border:1px solid #999}.note{color:#666;font-size:11px;line-height:1.4}.vis-tooltip{white-space:pre-line!important;max-width:520px;line-height:1.35}@media(max-width:850px){.page{grid-template-columns:1fr;height:auto}.sidebar{border-right:none;border-bottom:1px solid #ddd}.main{height:auto}#network{height:720px}}
</style>
</head>
<body><div class="page"><aside class="sidebar">
<h1>Frame-segment network</h1>
<h2>Model</h2><select id="modelFilter">__MODEL_OPTIONS__<option value="ALL">All models</option></select>
<h2>Document</h2><select id="docFilter"><option value="ALL">All documents</option>__DOC_OPTIONS__</select>
<h2>Year range</h2><label>From</label><input id="yearMin" type="number" value="__YEAR_MIN__"><label>To</label><input id="yearMax" type="number" value="__YEAR_MAX__">
<h2>Number of frames</h2><div class="small-actions"><button data-check-all="frameLabelOptions">All</button><button data-check-none="frameLabelOptions">None</button></div><div id="frameLabelOptions" class="check-group">__LABEL_CHECKS__</div>
<h2>Frame name</h2><div class="small-actions"><button data-check-all="frameNameOptions">All</button><button data-check-none="frameNameOptions">None</button></div><div id="frameNameOptions" class="check-group">__FRAME_CHECKS__</div>
<h2>Polarity</h2><select id="polarityFilter"><option value="ALL">All polarities</option>__POLARITY_OPTIONS__</select>
<h2>Relevance to original topic</h2><select id="relevanceFilter"><option value="ALL">All</option>__RELEVANCE_OPTIONS__</select>
<h2>Topic type</h2><div class="small-actions"><button data-check-all="topicTypeOptions">All</button><button data-check-none="topicTypeOptions">None</button></div><div id="topicTypeOptions" class="check-group">__TOPIC_CHECKS__</div>
<h2>Gender mentioned</h2><select id="genderFilter"><option value="ALL">All</option>__GENDER_OPTIONS__</select>
<h2>Network density</h2><label>Minimum frame degree</label><input id="minFrameDegree" type="number" min="1" value="1" step="1">
<h2>Node display</h2><label class="inline-check"><input id="showSegmentLabels" type="checkbox">Show segment labels</label><label class="inline-check"><input id="showMetadataHover" type="checkbox">Show extended metadata on hover</label>
<button id="resetFilters" style="width:100%;margin-top:10px">Reset filters</button>
<h2>Physics</h2><div class="physics-grid"><button data-speed="stop">Stop</button><button data-speed="slow">Slow</button><button data-speed="normal">Normal</button><button data-speed="rapid">Rapid</button></div>
<h2>Legend</h2><div id="legend"></div><div class="legend-item"><span class="swatch" style="background:#ddd;border-radius:3px"></span>Document segment</div><div class="legend-item"><span class="swatch" style="background:#f2f2f2;border-radius:0;transform:rotate(45deg)"></span>NOT_FRAMED</div>
<h2>Selected node</h2><div id="details" class="details">Click a node to inspect it.</div><p class="note">Node size reflects degree centrality in the currently filtered graph.</p>
</aside><main class="main"><div id="stats" class="stats"></div><div id="network"></div></main></div>
<script>
const segmentData=__SEGMENT_JSON__,edgeData=__EDGE_JSON__,frameColors=__FRAME_COLORS_JSON__;const MAX_EDGES=__MAX_EDGES__,MAX_VISIBLE_SEGMENTS=__MAX_VISIBLE_SEGMENTS__;
let network=null,nodesDataSet=null,edgesDataSet=null;
const $=id=>document.getElementById(id),modelFilter=$("modelFilter"),docFilter=$("docFilter"),yearMinInput=$("yearMin"),yearMaxInput=$("yearMax"),polarityFilter=$("polarityFilter"),relevanceFilter=$("relevanceFilter"),genderFilter=$("genderFilter"),minFrameDegreeInput=$("minFrameDegree"),showSegmentLabelsInput=$("showSegmentLabels"),showMetadataHoverInput=$("showMetadataHover"),stats=$("stats"),details=$("details"),legend=$("legend");
function esc(v){return String(v??"").replaceAll("&","&amp;").replaceAll("<","&lt;").replaceAll(">","&gt;").replaceAll('"',"&quot;").replaceAll("'","&#039;")}
Object.entries(frameColors).forEach(([f,c])=>{const r=document.createElement("div");r.className="legend-item";r.innerHTML='<span class="swatch" style="background:'+c+'"></span>'+esc(f);legend.appendChild(r)});
function checked(id){return new Set(Array.from(document.querySelectorAll("#"+id+" input[type=checkbox]:checked")).map(x=>x.value))}
function intersects(a,s){return a.some(x=>s.has(x))}
function filteredSegments(){const labs=checked("frameLabelOptions"),topics=checked("topicTypeOptions"),m=modelFilter.value,d=docFilter.value,rel=relevanceFilter.value,g=genderFilter.value;let ymin=+yearMinInput.value,ymax=+yearMaxInput.value;if(ymin>ymax)[ymin,ymax]=[ymax,ymin];return segmentData.filter(x=>{if(m!=="ALL"&&x.model!==m)return false;if(d!=="ALL"&&x.doc_id!==d)return false;if(x.year!==null&&(x.year<ymin||x.year>ymax))return false;if(!labs.has(x.frame_label))return false;if(rel!=="ALL"&&x.relevance_to_original_topic!==rel)return false;if(g!=="ALL"&&x.gender_mentioned!==g)return false;const miss=!x.topic_types||!x.topic_types.length;if(!topics.size)return false;if(miss){if(!topics.has("__MISSING__"))return false}else if(!intersects(x.topic_types,topics))return false;return true})}
function filteredEdgeRows(keys){const fs=checked("frameNameOptions"),p=polarityFilter.value;return edgeData.filter(e=>keys.has(e.model+"::"+e.segment_id)&&fs.size&&fs.has(e.frame_name)&&(p==="ALL"||e.polarity===p))}
function tip(meta,d){let t="Document: "+meta.doc_id+"\nSegment: "+meta.segment_index+"\nYear: "+(meta.year??"NA")+"\nModel: "+meta.model+"\nNumber of frames: "+meta.frame_label+"\nDegree: "+d;if(!showMetadataHoverInput.checked)return t;if(meta.title)t+="\n\nTitle:\n"+meta.title;if(meta.relevance_to_original_topic)t+="\n\nRelevance:\n"+meta.relevance_to_original_topic;if(meta.topic_types?.length)t+="\n\nTopic type:\n"+meta.topic_types.join(", ");if(meta.gender_mentioned)t+="\n\nGender mentioned:\n"+meta.gender_mentioned;if(meta.frame_evidence)t+="\n\nFrame evidence:\n"+meta.frame_evidence;if(meta.frame_explanation)t+="\n\nFrame explanation:\n"+meta.frame_explanation;return t}
function detailsSegment(m,d){const rows=[["Title",m.title||"NA"],["Document",m.doc_id],["Segment",m.segment_index],["Year",m.year??"NA"],["Model",m.model],["Number of frames",m.frame_label],["Topic type",m.topic_types?.length?m.topic_types.join(", "):"NA"],["Relevance",m.relevance_to_original_topic||"NA"],["Gender mentioned",m.gender_mentioned||"NA"],["Frame evidence",m.frame_evidence||"NA"],["Frame explanation",m.frame_explanation||"NA"],["Degree centrality",d]];details.innerHTML=rows.map(([k,v])=>"<b>"+esc(k)+"</b><br>"+esc(v)).join("<br><br>")}
function buildGraph(){stats.textContent="Building network...";const segs=filteredSegments(),segMap=new Map();segs.forEach(s=>segMap.set(s.model+"::"+s.segment_id,s));const keys=new Set(segMap.keys()),rows=filteredEdgeRows(keys),edgeMap=new Map(),frameMeta=new Map();rows.forEach(r=>{const s="SEG::"+r.model+"::"+r.segment_id,f="FRAME::"+r.frame_type,id=s+"||"+f;edgeMap.set(id,{id,from:s,to:f,color:{color:frameColors[r.frame_name]||"#888",opacity:.42},width:1});frameMeta.set(f,{type:"frame",frame_type:r.frame_type,frame_name:r.frame_name,polarity:r.polarity})});const edges=[...edgeMap.values()],rawDeg=new Map();edges.forEach(e=>{rawDeg.set(e.from,(rawDeg.get(e.from)||0)+1);rawDeg.set(e.to,(rawDeg.get(e.to)||0)+1)});const minD=Math.max(1,+minFrameDegreeInput.value||1),allowed=new Set([...frameMeta.keys()].filter(id=>(rawDeg.get(id)||0)>=minD)),fEdges=edges.filter(e=>allowed.has(e.to)),deg=new Map();fEdges.forEach(e=>{deg.set(e.from,(deg.get(e.from)||0)+1);deg.set(e.to,(deg.get(e.to)||0)+1)});const nodes=[];segMap.forEach((m,key)=>{const id="SEG::"+key,d=deg.get(id)||0,nf=m.frame_label==="NOT_FRAMED";if(d===0&&!nf)return;nodes.push({id,label:showSegmentLabelsInput.checked?m.doc_id+" / "+m.segment_index:"",shape:nf?"diamond":"box",size:d?8+13*Math.sqrt(d/6):8,value:d,color:{background:nf?"#f2f2f2":"#ddd",border:"#999",highlight:{background:"#eee",border:"#555"}},font:{size:10},title:tip(m,d),metadata:{...m,type:"segment"}})});const maxFD=Math.max(1,...[...allowed].map(id=>deg.get(id)||0));frameMeta.forEach((m,id)=>{if(!allowed.has(id))return;const d=deg.get(id)||0;if(!d)return;const c=frameColors[m.frame_name]||"#888";nodes.push({id,label:m.frame_type,shape:"dot",size:16+30*Math.sqrt(d/maxFD),value:d,color:{background:c,border:"#444",highlight:{background:c,border:"#000"}},font:{size:14},title:"Frame: "+m.frame_type+"\nDegree: "+d,metadata:m})});const nSeg=nodes.filter(n=>n.metadata.type==="segment").length,nFrame=nodes.filter(n=>n.metadata.type==="frame").length;if(fEdges.length>MAX_EDGES||nSeg>MAX_VISIBLE_SEGMENTS){if(network){network.destroy();network=null}$("network").innerHTML="<div style='padding:40px;text-align:center;color:#666'><b>Network too large to render.</b><br><br>"+fEdges.length.toLocaleString()+" links and "+nSeg.toLocaleString()+" visible segments remain after filtering.<br>Select one model/document, narrow years, or increase minimum frame degree.</div>";stats.textContent="Please narrow filters";return}const c=$("network");c.innerHTML="";nodesDataSet=new vis.DataSet(nodes);edgesDataSet=new vis.DataSet(fEdges);const opts={autoResize:true,layout:{improvedLayout:false},interaction:{hover:true,tooltipDelay:120,multiselect:true,navigationButtons:true,keyboard:true},physics:{enabled:true,solver:"forceAtlas2Based",forceAtlas2Based:{gravitationalConstant:-30,centralGravity:.01,springLength:100,springConstant:.04,damping:.6,avoidOverlap:.5},stabilization:{enabled:true,iterations:50}},edges:{smooth:{enabled:true,type:"continuous"},selectionWidth:2},nodes:{borderWidth:1}};if(network)network.destroy();network=new vis.Network(c,{nodes:nodesDataSet,edges:edgesDataSet},opts);network.on("selectNode",p=>{if(!p.nodes.length)return;const n=nodesDataSet.get(p.nodes[0]);if(n.metadata.type==="frame")details.innerHTML="<b>Frame</b><br>"+esc(n.metadata.frame_type)+"<br><br><b>Frame name</b><br>"+esc(n.metadata.frame_name)+"<br><br><b>Polarity</b><br>"+esc(n.metadata.polarity)+"<br><br><b>Degree centrality</b><br>"+esc(n.value);else detailsSegment(n.metadata,n.value)});network.on("deselectNode",()=>details.textContent="Click a node to inspect it.");stats.textContent=nSeg.toLocaleString()+" segments · "+nFrame.toLocaleString()+" frame types · "+fEdges.length.toLocaleString()+" links"}
[modelFilter,docFilter,polarityFilter,relevanceFilter,genderFilter,yearMinInput,yearMaxInput,minFrameDegreeInput,showSegmentLabelsInput,showMetadataHoverInput].forEach(x=>x.addEventListener("change",buildGraph));["frameNameOptions","frameLabelOptions","topicTypeOptions"].forEach(id=>document.querySelectorAll("#"+id+" input[type=checkbox]").forEach(x=>x.addEventListener("change",buildGraph)));document.querySelectorAll("[data-check-all]").forEach(b=>b.addEventListener("click",()=>{document.querySelectorAll("#"+b.dataset.checkAll+" input[type=checkbox]").forEach(x=>x.checked=true);buildGraph()}));document.querySelectorAll("[data-check-none]").forEach(b=>b.addEventListener("click",()=>{document.querySelectorAll("#"+b.dataset.checkNone+" input[type=checkbox]").forEach(x=>x.checked=false);buildGraph()}));
$("resetFilters").addEventListener("click",()=>{modelFilter.value=__DEFAULT_MODEL_JSON__;docFilter.value="ALL";polarityFilter.value="ALL";relevanceFilter.value="ALL";genderFilter.value="ALL";yearMinInput.value="__YEAR_MIN__";yearMaxInput.value="__YEAR_MAX__";minFrameDegreeInput.value="1";showSegmentLabelsInput.checked=false;showMetadataHoverInput.checked=false;["frameNameOptions","frameLabelOptions","topicTypeOptions"].forEach(id=>document.querySelectorAll("#"+id+" input[type=checkbox]").forEach(x=>x.checked=true));buildGraph()});document.querySelectorAll("[data-speed]").forEach(b=>b.addEventListener("click",()=>{if(!network)return;const s=b.dataset.speed;if(s==="stop"){network.setOptions({physics:{enabled:false}});return}let damping=.6,timestep=.5;if(s==="slow"){damping=.8;timestep=.25}if(s==="rapid"){damping=.2;timestep=1}network.setOptions({physics:{enabled:true,timestep,forceAtlas2Based:{damping}}});network.startSimulation()}));if(typeof vis==="undefined")stats.innerHTML="<b>Error:</b> vis-network did not load. Check internet/browser console.";else buildGraph();
</script></body></html>"""

    replacements = {
        "__SEGMENT_JSON__": _js_json(segment_records),
        "__EDGE_JSON__": _js_json(edge_records),
        "__FRAME_COLORS_JSON__": _js_json(FRAME_COLORS),
        "__MODEL_OPTIONS__": model_options,
        "__DOC_OPTIONS__": doc_options,
        "__POLARITY_OPTIONS__": polarity_options,
        "__RELEVANCE_OPTIONS__": relevance_options,
        "__GENDER_OPTIONS__": gender_options,
        "__FRAME_CHECKS__": frame_checks,
        "__LABEL_CHECKS__": label_checks,
        "__TOPIC_CHECKS__": topic_checks,
        "__YEAR_MIN__": str(year_min),
        "__YEAR_MAX__": str(year_max),
        "__MAX_EDGES__": str(int(max_edges)),
        "__MAX_VISIBLE_SEGMENTS__": str(int(max_visible_segments)),
        "__DEFAULT_MODEL_JSON__": json.dumps(models[0] if models else "ALL"),
    }

    html_text = template
    for key, value in replacements.items():
        html_text = html_text.replace(key, value)

    output_file = Path(output_file)
    output_file.write_text(html_text, encoding="utf-8")
    return output_file

In [ ]:
# ============================================================
# 5. Generate the HTML
# ============================================================
output_path = build_network_html(segment_df, edge_df)
print("Saved:", output_path.resolve())
print(f"Size: {output_path.stat().st_size / 1_000_000:.1f} MB")

## Preview and publish

For local testing, serve the project directory rather than opening the HTML with a `file://` URL:

```bash
python3 -m http.server 8000
```

Then open `http://localhost:8000/frame_segment_network.html`.

For GitHub Pages, commit `frame_segment_network.html` to the published branch/folder. The page loads the pinned `vis-network@9.1.9` library from `unpkg.com`.

### Filter behavior

- **Frame name**, **Number of frames**, and **Topic type** support multiple selections.
- Multiple selections within one filter use **OR** logic; different filter families combine with **AND** logic.
- `topic_type` is split on semicolons.
- `NOT_FRAMED` segments are retained and appear as isolated diamond nodes.
- Extended metadata (`title`, `frame_evidence`, `frame_explanation`, etc.) can be enabled for hover; it is always available in the click-details panel.